# Caso B · 05 Validación 24h — walk-forward y métricas por horizonte

> _Tutorial · Caso de uso: **B — Forecast consumo 24h** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Validar el modelo seleccionado con walk-forward y reportar métricas por horizonte (1h, 6h, 12h, 24h).


## 2. Qué se aprende

- Walk-forward: re-entrenar al avanzar en el tiempo.
- Cómo el error crece con el horizonte.
- Cuándo dejar de re-entrenar (concept drift).


## 3. Contexto del caso de uso

El curso evalúa el modelo a 24h. La validación walk-forward es la única que mantiene la temporalidad.


## 4. Relación con CENTINELA+

Producción re-entrena cada noche. El reporte aquí imita ese ciclo.


## 5. Relación con Medallion

Oro: dataset de validación + reporte.


## 6. Datos de entrada

Features oro del Caso B.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Reusamos el feature dataset.


In [ ]:
df, _ = mocks.make_bdg2_education_subset()
df = df[df.building_id == df.building_id.unique()[0]].set_index("timestamp")
X = pd.DataFrame(index=df.index)
X["y"] = df["power_kw"]
X["t_outdoor"] = df["t_outdoor"]
X["ghi"] = df["ghi"]
for lag in [1, 24, 168]:
    X[f"lag_{lag}h"] = df["power_kw"].shift(lag)
X = X.dropna()
print("Filas:", len(X))


## 10. Exploración paso a paso

Splits semanales con incremento.


In [ ]:
def walk_forward_split(idx, train_weeks=8, step_hours=24, n_folds=8):
    folds = []
    cur = train_weeks * 7 * 24
    while cur + step_hours < len(idx) and len(folds) < n_folds:
        folds.append((idx[:cur], idx[cur : cur + step_hours]))
        cur += step_hours
    return folds

folds = walk_forward_split(X.index)
print(f"{len(folds)} folds; primer test = {folds[0][1][0]} → {folds[0][1][-1]}")


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Loop de validación.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

def metricas(y, p):
    return {
        "MAE": float(np.mean(np.abs(y - p))),
        "RMSE": float(np.sqrt(np.mean((y - p) ** 2))),
        "MAPE_%": float(np.mean(np.abs((y - p) / np.maximum(np.abs(y), 1e-3))) * 100),
    }

resultados = []
for tr_idx, te_idx in folds:
    X_tr, y_tr = X.loc[tr_idx].drop(columns=["y"]), X.loc[tr_idx]["y"]
    X_te, y_te = X.loc[te_idx].drop(columns=["y"]), X.loc[te_idx]["y"]
    m = RandomForestRegressor(n_estimators=120, random_state=SEED).fit(X_tr, y_tr)
    p = m.predict(X_te)
    for hours_ahead in [1, 6, 12, 23]:
        if hours_ahead < len(y_te):
            r = metricas(y_te.iloc[[hours_ahead]].values, p[hours_ahead : hours_ahead + 1])
            r.update({"fold": tr_idx[-1], "horizon": hours_ahead + 1})
            resultados.append(r)

res = pd.DataFrame(resultados)
res.head()


## 13. Visualizaciones explicativas

MAE medio por horizonte.


In [ ]:
mae_por_h = res.groupby("horizon")["MAE"].mean()
mae_por_h.plot.bar(color="#3F51B5", figsize=(7, 3))
plt.title("MAE medio por horizonte (h)")
plt.ylabel("kW")
plt.tight_layout()


## 14. Validaciones

El error a 24h debe ser <= 2× el error a 1h en este mock.


In [ ]:
mae_1h = mae_por_h.iloc[0]
mae_24h = mae_por_h.iloc[-1]
print({"mae_1h": mae_1h, "mae_24h": mae_24h, "ratio": mae_24h / mae_1h})


## 15. Errores comunes

1. Reportar solo el último fold (varianza alta).
2. Comparar MAE entre datasets sin normalizar.
3. No incluir un naive comparable.


## 16. Ejercicios propuestos

1. Añade walk-forward semanal en lugar de diario.
2. Implementa drift detection con `EDDM`.
3. Reporta intervalos de confianza con bootstrap.


## 17. Cómo se reutiliza con datos reales

Idéntico — el iterador `walk_forward_split` solo usa el índice temporal.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `03_case_C_hvac_anomaly_detection/01_eda_hvac_fdd.ipynb`.
- Documento web del caso: `docs/validation/ml-validation.md`.
